In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path
import torch
import numpy as np

logging.basicConfig(level=logging.INFO)

sys.path.insert(0, str(Path.cwd().parents[0]))
sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

# Experiment Setup

In [ ]:
EXPERIMENT_NAME = "real_loss_sweep"
RESULTS_DIR = Path("../segm/results") / EXPERIMENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 5
RUN_BASE_SEED = 12345

print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Number of runs per loss: {N_RUNS}")

In [ ]:
TEST_ROOT = Path("../segm/real_data/test")
REAL_ROOT = Path("../segm/real_data/labeled_wo_test")
SYNTH_ROOT = Path("../segm/fin_data/gen_basic")

# Data Loading

In [ ]:
from precipseg import data

LOAD_SYNTH_MODE = "basic"  # "basic" or "clustered"

# only used for "basic"
LOAD_NUM_SYNTH = 0

# only used for "clustered"
CLUSTERS_BASE_COUNTS = (10, 13, 7, 9, 6)  
CLUSTERS_MULTIPLICATOR = 5
CLUSTERS_DIR_PREFIX = "cl"  # assumes the clustered images were generated like *{prefix}{i}/img.png

In [ ]:
# test
test_img_paths, test_label_paths = data.load_paths(TEST_ROOT)
test_imgs = [data.read_image_normalized(p) for p in test_img_paths]
test_labels = [data.read_image_normalized(p) for p in test_label_paths]

# train real
real_img_paths, real_label_paths = data.load_paths(REAL_ROOT)

# train synthetic
if LOAD_SYNTH_MODE == "basic":
    synth_img_paths, synth_label_paths = data.load_paths(SYNTH_ROOT, LOAD_NUM_SYNTH)
elif LOAD_SYNTH_MODE == "clustered":
    synth_img_paths, synth_label_paths = data.load_clustered(SYNTH_ROOT, CLUSTERS_BASE_COUNTS, CLUSTERS_MULTIPLICATOR)
else:
    raise ValueError(f"Unknown LOAD_SYNTH_MODE={LOAD_SYNTH_MODE!r}")


train_img_paths = real_img_paths + synth_img_paths
train_label_paths = real_label_paths + synth_label_paths

print(f"Train: {len(real_img_paths)} real + {len(synth_img_paths)} synthetic ({LOAD_SYNTH_MODE}).")
print(f"Test: {len(test_img_paths)}.")

In [ ]:
from precipseg import data

PATCH_SIZE = 256
X_CHANNELS = 3
BATCH_SIZE = 16
VAL_SIZE = 0.25

train_dl, val_dl = data.build_dataloaders(
    train_img_paths, train_label_paths,
    patch_size=PATCH_SIZE,
    x_channels=X_CHANNELS,
    batch_size=BATCH_SIZE,
    val_size=VAL_SIZE,
)

In [ ]:
from precipseg.data import show_augmented_triple

show_augmented_triple(n_show=5, train_dl=train_dl)

# Loss Selection

In [ ]:
from precipseg.losses import available_losses
from itertools import product

print("Available losses:", available_losses())

LOSS_PARAM_GRID = {
    "focal": dict(
        alpha=[0.3, 0.5, 0.7],
        gamma=[1.5, 2.0, 2.5],
    ),
    "focal_dice": dict(
        alpha=[0.2, 0.4, 0.6],
        gamma=[1.5, 2.0, 2.5],
        smooth=[1e-6],
    ),
    "focal_tversky": dict(
        alpha=[0.3, 0.5, 0.7],
        beta=[0.7, 0.5, 0.3],
        gamma=[1.3],
        bce_weight=[0.3],
        smooth=[1e-6],
    ),
}


LOSSES = []

for loss_name, grid in LOSS_PARAM_GRID.items():
    for values in product(*grid.values()):
        LOSSES.append({"loss_name": loss_name, "params": dict(zip(grid.keys(), values))})

print(f"Number of loss configurations: {len(LOSSES)}")

# Training

In [ ]:
from precipseg.training import load_run_json, has_required_metrics, run_one
from precipseg.training import ImgData


img_data = ImgData(
    train_img_paths=train_img_paths,
    train_label_paths=train_label_paths,
    test_imgs=test_imgs,
    test_labels=test_labels,
    real_imgs=real_img_paths,
    synth_imgs=synth_img_paths,
    synth_root=SYNTH_ROOT,
    patch_size=PATCH_SIZE,
    batch_size=BATCH_SIZE,
    val_size=VAL_SIZE,
    split_seed=123,
)


THRESHOLDS = np.linspace(0.10, 0.95, 10)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for cfg in LOSSES:
    loss_name = cfg["loss_name"]
    loss_params = cfg["params"]

    tag = f"{loss_name}-" + "-".join(f"{k}{v}" for k, v in loss_params.items())
    runs_dir = RESULTS_DIR / tag
    runs_dir.mkdir(parents=True, exist_ok=True)

    runs = []
    for i in range(N_RUNS):
        run_seed = RUN_BASE_SEED + i
        run_dir = runs_dir / f"seed_{run_seed}"
        run_json = run_dir / "run.json"

        if run_json.exists():
            cached = load_run_json(run_json)
            if has_required_metrics(cached):
                runs.append(cached)
                continue

        runs.append(
            run_one(
                loss_name=loss_name,
                loss_params=loss_params,
                run_seed=run_seed,
                runs_dir=runs_dir,
                x_channels=X_CHANNELS,
                device=DEVICE,
                thresholds=THRESHOLDS,
                max_images=6,
                data=img_data,
            )
        )

# Results Evaluation

In [ ]:
from precipseg.evaluation import run_full_evaluation

root_data_dir = ("../segm/results/real_loss_sweep/")

# This creates /global_reports and fills each experiment folder
run_full_evaluation(root_data_dir, metrics_file_name="metrics_recomputed_new.txt", main_metric="dice", n_extremes=10)

## Granular Evaluation

In [ ]:
from precipseg.evaluation import collect_runs, prepare_master_df

root_data_dir = ("../segm/results/new_real_loss_sweep/focal_dice-alpha0.6-gamma1.5-smooth1e-06/")

grouped_runs, config_paths = collect_runs(root_data_dir, metrics_file_name="metrics.txt")
master_df = prepare_master_df(grouped_runs, main_metric="dice", config_paths=config_paths)

In [ ]:
from precipseg.evaluation import save_main_metric, save_full_metrics, save_extremes

global_out = Path("./custom_analysis")
global_out.mkdir(exist_ok=True)

# Save the leaderboard + boxplot
leaderboard_df = save_main_metric(master_df, main_metric="dice", out_dir=global_out, plot=True)
leaderboard_df

In [ ]:
# The full diagnostic table (Mean ± Std)
full_metrics_df = save_full_metrics(master_df, main_metric="dice", out_dir=global_out)
full_metrics_df

In [ ]:
# Audit the x best/worst performing seeds (mode="top", "bottom", "both")
extremes_df = save_extremes(master_df, main_metric="dice", n=5, out_dir=global_out, mode="bottom")
extremes_df

In [ ]:
from precipseg.evaluation import save_individual_reports

# This will go into each experiment folder, create the variance plot and save the all_seeds_summary.csv
save_individual_reports(master_df, plot=True)